# 03_repeat_seeds.ipynb

Chạy lặp Swin-Tiny, ResNet-50 và ConvNeXt-Tiny với seed 43 và 44, cùng giao thức với lần chạy seed 42: train, chọn checkpoint bằng macro F1 validation, rồi đo F1 sạch và mức giảm dưới năm suy giảm cố định trên validation (cùng quy trình `03_select_reference_model.ipynb`). Cuối cùng gộp với seed 42 thành bảng trung bình và độ lệch chuẩn theo model. Notebook chạy trên Kaggle với GPU. Tập test không mở.

Danh sách seed chốt trước khi chạy: 42, 43, 44. Không bỏ seed nào theo kết quả. ResNet-50 được chạy lặp vì là ứng viên sát nút của ConvNeXt-Tiny khi chọn reference (F1 sạch chênh 0,0055). MobileNetV3-Small không chạy lặp vì đã bị loại khỏi nhóm ứng viên ở bước F1 sạch (kém 0,048), xem `reports/baselines/README.md`. Thời gian ước tính trên Tesla T4 (theo thời gian các lần chạy seed 42): Swin-Tiny khoảng 14 phút mỗi lần, ResNet-50 khoảng 29 phút, ConvNeXt-Tiny khoảng 52 phút, tổng khoảng 3 đến 3,5 giờ.

## 1. Môi trường

Internet để On. Accelerator chọn GPU. Dataset input là `cthng123/durian-leaf-disease-dataset`. Phần train không cần checkpoint cũ.

Seed 42 để tổng hợp lấy từ `reports/baselines/reference_selection_with_swin.json` trong repo (nhánh `baseline-Swin-Tiny`).

In [1]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "baseline-Swin-Tiny"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

Cloning into '/kaggle/working/durian-leaf-disease-robustness'...


Nhánh: baseline-Swin-Tiny
Commit: 3f3c7b6
Ảnh: /kaggle/working/durian-leaf-disease-robustness/datasets/durian-ldd/DLD_FinalDataset_224_spit


## 2. Cấu hình

Mỗi file `configs/train_<model>_seed<N>.yaml` giống file gốc, chỉ thêm khóa `seed` ghi đè seed trong `configs/pipeline.yaml`. Split dữ liệu và seed của suy giảm không đổi, nên mọi model được chấm trên đúng cùng ảnh. Thứ tự chạy từ rẻ đến tốn: Swin-Tiny, ResNet-50, ConvNeXt-Tiny.

In [2]:
from src.training.run import load_finetune_configs

SEEDS = [43, 44]
MODELS = ["swin_tiny", "resnet50", "convnext_tiny"]

for seed in SEEDS:
    for model in MODELS:
        train_cfg, pipeline_cfg = load_finetune_configs(f"configs/train_{model}_seed{seed}.yaml")
        assert pipeline_cfg["seed"] == seed and train_cfg["model_name"] == model
        print(f"{model} seed {pipeline_cfg['seed']}: lr {train_cfg['learning_rate']}, epoch tối đa {train_cfg['epochs']}, patience {train_cfg['patience']}")
print("Tập test: không mở")

swin_tiny seed 43: lr 0.0002, epoch tối đa 30, patience 7
resnet50 seed 43: lr 0.0002, epoch tối đa 30, patience 7
convnext_tiny seed 43: lr 0.0002, epoch tối đa 30, patience 7
swin_tiny seed 44: lr 0.0002, epoch tối đa 30, patience 7
resnet50 seed 44: lr 0.0002, epoch tối đa 30, patience 7
convnext_tiny seed 44: lr 0.0002, epoch tối đa 30, patience 7
Tập test: không mở


## 3. Train

Mỗi lượt gọi `src.training.run.main`, đúng đường chạy của notebook `03_train_<model>`. Checkpoint, JSON và hình ghi vào `outputs/`; JSON và hình được chép sang `reports/baselines/<model>/` để đo độ bền ở mục 4 và để commit. Nếu phiên bị ngắt giữa chừng, chạy lại từ cell này: lượt nào đã có checkpoint thì bị bỏ qua (chỉ đúng khi cùng một phiên còn file, vì phiên mới bắt đầu từ máy sạch).

In [3]:
import shutil

from src.training.run import ROOT, main

runs = {}
for seed in SEEDS:
    for model in MODELS:
        done = ROOT / "outputs/checkpoints" / f"{model}_seed{seed}.pt"
        if done.is_file():
            print(f"{model} seed {seed}: đã có checkpoint, bỏ qua", flush=True)
            continue
        print(f"=== {model} seed {seed} ===", flush=True)
        paths = main(f"configs/train_{model}_seed{seed}.yaml")
        runs[(model, seed)] = paths
        target = ROOT / "reports/baselines" / model
        target.mkdir(parents=True, exist_ok=True)
        for key in ("metrics", "confusion_matrix", "history"):
            shutil.copy(paths[key], target / Path(paths[key]).name)
print(sorted(runs))

=== swin_tiny seed 43 ===


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/30  train_loss=0.6368  val_loss=0.3313  val_macro_f1=0.9108  val_balanced_accuracy=0.9146  333.1s  checkpoint
Epoch 02/30  train_loss=0.2570  val_loss=0.1160  val_macro_f1=0.9666  val_balanced_accuracy=0.9662  57.9s  checkpoint
Epoch 03/30  train_loss=0.1656  val_loss=0.1955  val_macro_f1=0.9314  val_balanced_accuracy=0.9289  56.8s
Epoch 04/30  train_loss=0.1665  val_loss=0.1818  val_macro_f1=0.9375  val_balanced_accuracy=0.9393  56.3s
Epoch 05/30  train_loss=0.1290  val_loss=0.1308  val_macro_f1=0.9587  val_balanced_accuracy=0.9582  56.9s
Epoch 06/30  train_loss=0.1083  val_loss=0.2412  val_macro_f1=0.9371  val_balanced_accuracy=0.9404  56.4s
Epoch 07/30  train_loss=0.1640  val_loss=0.1523  val_macro_f1=0.9499  val_balanced_accuracy=0.9496  55.7s
Epoch 08/30  train_loss=0.0809  val_loss=0.0915  val_macro_f1=0.9789  val_balanced_accuracy=0.9776  56.6s  checkpoint
Epoch 09/30  train_loss=0.0538  val_loss=0.1181  val_macro_f1=0.9725  val_balanced_accuracy=0.9727  56.5s
Epoch 10/

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/30  train_loss=1.4965  val_loss=1.2629  val_macro_f1=0.4939  val_balanced_accuracy=0.5150  49.3s  checkpoint
Epoch 02/30  train_loss=0.8774  val_loss=0.4902  val_macro_f1=0.8565  val_balanced_accuracy=0.8599  49.8s  checkpoint
Epoch 03/30  train_loss=0.3894  val_loss=0.3330  val_macro_f1=0.8896  val_balanced_accuracy=0.8887  50.4s  checkpoint
Epoch 04/30  train_loss=0.2573  val_loss=0.1971  val_macro_f1=0.9400  val_balanced_accuracy=0.9401  49.8s  checkpoint
Epoch 05/30  train_loss=0.1858  val_loss=0.1701  val_macro_f1=0.9472  val_balanced_accuracy=0.9475  49.2s  checkpoint
Epoch 06/30  train_loss=0.1540  val_loss=0.1348  val_macro_f1=0.9611  val_balanced_accuracy=0.9608  49.9s  checkpoint
Epoch 07/30  train_loss=0.1222  val_loss=0.1491  val_macro_f1=0.9458  val_balanced_accuracy=0.9455  49.6s
Epoch 08/30  train_loss=0.1125  val_loss=0.1283  val_macro_f1=0.9587  val_balanced_accuracy=0.9584  50.2s
Epoch 09/30  train_loss=0.1061  val_loss=0.1221  val_macro_f1=0.9624  val_balanc

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/30  train_loss=1.6927  val_loss=2.0420  val_macro_f1=0.1495  val_balanced_accuracy=0.2466  96.3s  checkpoint
Epoch 02/30  train_loss=1.4956  val_loss=1.4796  val_macro_f1=0.2157  val_balanced_accuracy=0.3011  85.1s  checkpoint
Epoch 03/30  train_loss=1.3654  val_loss=1.2583  val_macro_f1=0.4598  val_balanced_accuracy=0.4807  85.7s  checkpoint
Epoch 04/30  train_loss=1.1141  val_loss=1.2891  val_macro_f1=0.4430  val_balanced_accuracy=0.5039  85.2s
Epoch 05/30  train_loss=0.6749  val_loss=0.4377  val_macro_f1=0.8288  val_balanced_accuracy=0.8254  86.2s  checkpoint
Epoch 06/30  train_loss=0.3530  val_loss=0.5956  val_macro_f1=0.7387  val_balanced_accuracy=0.7809  86.0s
Epoch 07/30  train_loss=0.4889  val_loss=0.3014  val_macro_f1=0.9019  val_balanced_accuracy=0.8989  86.8s  checkpoint
Epoch 08/30  train_loss=0.1751  val_loss=0.4211  val_macro_f1=0.8489  val_balanced_accuracy=0.8523  86.5s
Epoch 09/30  train_loss=0.1611  val_loss=0.0880  val_macro_f1=0.9820  val_balanced_accuracy=

## 4. Độ bền trên validation

Đo từng seed bằng `configs/reference_selection_seed<N>.yaml`: cùng validation, cùng năm suy giảm (tối 0.5, sáng 1.5, tương phản 0.5, mờ σ 2, nhiễu 0.1) và cùng seed của phép đo (42) như lần đo gốc. Không dùng để chọn lại reference model, dòng `selection` trong kết quả chỉ để tham khảo.

In [4]:
from src.evaluation.reference import main as measure

CHECKPOINT_DIR = ROOT / "outputs/checkpoints"
records = {}
for seed in SEEDS:
    records[seed] = measure(f"configs/reference_selection_seed{seed}.yaml", CHECKPOINT_DIR)

resnet50: clean 0.9763  corrupted 0.7365  6.527 ms/ảnh
convnext_tiny: clean 0.9820  corrupted 0.7274  7.134 ms/ảnh
swin_tiny: clean 0.9789  corrupted 0.8494  12.818 ms/ảnh
Reference model: swin_tiny (macro F1 sạch ngang nhau, mức giảm dưới suy giảm thấp nhất)
resnet50: clean 0.9724  corrupted 0.6976  6.731 ms/ảnh
convnext_tiny: clean 0.9743  corrupted 0.7788  6.289 ms/ảnh
swin_tiny: clean 0.9750  corrupted 0.8339  11.91 ms/ảnh
Reference model: swin_tiny (macro F1 sạch ngang nhau, mức giảm dưới suy giảm thấp nhất)


## 5. Tổng hợp qua seed

Seed 42 lấy từ `reference_selection_with_swin.json` (cùng quy trình đo). Độ lệch chuẩn tính trên ba điểm nên chỉ là dấu hiệu, không phải ước lượng chắc.

In [5]:
import json

import pandas as pd

base = json.loads((ROOT / "reports/baselines/reference_selection_with_swin.json").read_text(encoding="utf-8"))["models"]
sources = {42: base, **{seed: record["models"] for seed, record in records.items()}}
rows = []
for seed, models in sources.items():
    for model in MODELS:
        entry = models[model]
        rows.append({
            "model": model, "seed": seed,
            "clean_macro_f1": entry["clean"]["macro_f1"],
            **entry["corrupted"]["macro_f1"],
            "mean_corrupted": entry["corrupted"]["mean_macro_f1"],
            "degradation": entry["corrupted"]["degradation"],
        })
per_seed = pd.DataFrame(rows).sort_values(["model", "seed"]).reset_index(drop=True)
display(per_seed.round(4))

summary = per_seed.groupby("model")[["clean_macro_f1", "mean_corrupted", "degradation"]].agg(["mean", "std"]).round(4)
display(summary)

pivot = per_seed.pivot(index="seed", columns="model", values="degradation")
pivot["swin_minus_convnext"] = pivot["swin_tiny"] - pivot["convnext_tiny"]
pivot["resnet_minus_convnext"] = pivot["resnet50"] - pivot["convnext_tiny"]
print("Mức giảm theo seed (âm nghĩa là model đó giảm ít hơn ConvNeXt-Tiny):")
display(pivot.round(4))

,model,seed,clean_macro_f1,brightness_dark,brightness_bright,contrast_low,blur,noise,mean_corrupted,degradation
0,convnext_tiny,42,0.9775,0.8923,0.8508,0.8761,0.5770,0.6582,0.7709,0.2066
1,convnext_tiny,43,0.9820,0.8795,0.8932,0.8545,0.4971,0.5129,0.7274,0.2546
2,convnext_tiny,44,0.9743,0.9241,0.8615,0.8903,0.5687,0.6494,0.7788,0.1954
3,resnet50,42,0.9720,0.9089,0.9205,0.7345,0.4047,0.5678,0.7073,0.2647
4,resnet50,43,0.9763,0.9084,0.8386,0.8184,0.5020,0.6152,0.7365,0.2398
5,resnet50,44,0.9724,0.8875,0.8154,0.8870,0.2524,0.6453,0.6976,0.2748
6,swin_tiny,42,0.9664,0.9273,0.8769,0.9542,0.6529,0.7382,0.8299,0.1365
7,swin_tiny,43,0.9789,0.9466,0.8993,0.9588,0.6374,0.8050,0.8494,0.1294
8,swin_tiny,44,0.9750,0.9329,0.9108,0.9424,0.5895,0.7938,0.8339,0.1411


clean_macro_f1         mean_corrupted         degradation  \
                        mean     std           mean     std        mean   
model                                                                     
convnext_tiny         0.9779  0.0039         0.7590  0.0277      0.2189   
resnet50              0.9736  0.0024         0.7138  0.0203      0.2598   
swin_tiny             0.9734  0.0064         0.8377  0.0103      0.1357   

                       
                  std  
model                  
convnext_tiny  0.0314  
resnet50       0.0180  
swin_tiny      0.0059

Mức giảm theo seed (âm nghĩa là model đó giảm ít hơn ConvNeXt-Tiny):


model,convnext_tiny,resnet50,swin_tiny,swin_minus_convnext,resnet_minus_convnext
seed,,,,,
42,0.2066,0.2647,0.1365,-0.0701,0.0582
43,0.2546,0.2398,0.1294,-0.1251,-0.0148
44,0.1954,0.2748,0.1411,-0.0543,0.0794


## 6. Xuất kết quả

Bảng tổng hợp ghi vào `reports/baselines/seed_repeat_summary.csv`. Checkpoint, JSON, hình và kết quả đo độ bền được nén thành hai file zip để tải về. Checkpoint tải riêng, không đưa vào Git.

In [6]:
per_seed.to_csv(ROOT / "reports/baselines/seed_repeat_summary.csv", index=False)
shutil.make_archive("/kaggle/working/seed_runs_outputs", "zip", root_dir=ROOT, base_dir="outputs/checkpoints")
shutil.make_archive("/kaggle/working/seed_runs_reports", "zip", root_dir=ROOT, base_dir="reports/baselines")
print("Đã lưu:", sorted(path.name for path in (ROOT / "reports/baselines").glob("*seed*")))

Đã lưu: ['reference_selection_seed43.json', 'reference_selection_seed44.json', 'seed_repeat_summary.csv']
